In [34]:
from json import dump, load
from pandas import DataFrame, concat
from tqdm import tqdm
from copy import deepcopy
from pickle import dump
import numpy as np
import chython as ch

In [3]:
with open('old_data.json', 'r') as file1, open('new_data_smi_v3.json') as file2:
    data_old = load(file1)
    data_new = load(file2)

In [4]:
data_merge = deepcopy(data_new)
for cid, info in data_old.items():
    smi = info['smiles']
    values = deepcopy(info['ld50'])
    data_merge[smi] = data_merge.get(smi, []) + values

In [23]:
test = deepcopy(data_merge)
for smi, list_of_values in test.items():  # Удаление "полных" дублей
    new_values = []
    if len(list_of_values) == 1:
        continue
    else:
        generate_list = tuple(tuple(el.values()) for el in list_of_values)
        for indx, value in enumerate(generate_list):
            if generate_list.count(value) == 1:
                new_values.append(list_of_values[indx])
            elif (generate_list.count(value) > 1
                  and (list_of_values[indx] not in new_values)):
                new_values.append(list_of_values[indx])
    test[smi] = new_values

print(sum(len(el) for el in test.values()), sum(len(el) for el in data_merge.values()))

68495 76949


In [28]:
for values in test.values():
    for value in values.copy():
        match value['measure']:
            case 'mg/kg':
                continue
            case 'mg/kg-day':
                value['measure'] = 'mg/kg'
            case _:
                values.remove(value)

del test['NaN']
sum(len(el) for el in test.values())

66394

In [5]:
def data_cleaner(data: dict, key: str, val: str):
    data = deepcopy(data)
    for smi, values in deepcopy(data).items():
        for value in deepcopy(values):
            if str(value[key]).lower() != val.lower():
                data[smi].remove(value)
        if not len(data[smi]):
            del data[smi]
    
    return data

counter = lambda collection: sum(len(el) for el in collection.values())

In [31]:
with open('data_merge.json', 'r') as file:
    data = load(file)

In [16]:
len(data)

23728

In [32]:
min_data = {}
for smi, values in tqdm(data.items()):
    if values:
        value = min(values, key=lambda el: el["value"])
        min_data[smi] = value

  0%|          | 0/23726 [00:00<?, ?it/s]

100%|██████████| 23726/23726 [00:00<00:00, 64169.79it/s]


In [33]:
min_ld50 = DataFrame().from_dict({'SMILES': [smi for smi in min_data.keys()], 'LD50': [val['value'] for val in min_data.values()]})

In [ ]:
min_ld50_clear = DataFrame(columns=['SMILES', 'LD50'])
for _smiles, val in tqdm(zip(min_ld50['SMILES'], min_ld50['LD50'])):
    try:
        mol = ch.smiles(_smiles)
        mol.canonicalize()
        mol.clean_isotopes()
        if mol.check_valence():
            print(f'valerr: {_smiles}')
            continue
        
        min_ld50_clear.loc[len(min_ld50_clear)] = [str(mol), val]
    except Exception as ex:
        print(f'{ex}: {_smiles}')

0it [00:00, ?it/s]

105it [00:01, 83.96it/s]

valerr: [Al+3].[P-3]
valerr: [P-3].[P-3].[Ca++].[Ca++].[Ca++]
valerr: [Mg++].[Mg++].[Mg++].[P-3].[P-3]


253it [00:05, 31.37it/s]

valerr: [O--].[Cu+].[Cu+]


...

21173it [07:07, 16.25it/s]

valerr: O.O.O.O.[Pt++].[O-]S([O-])(=O)=O


23184it [08:04, 58.12it/s] 

valerr: [H][C@@](C)(O)C([H])(CCCCCC)[N]1#CN=C2C1=[NH+]C=NC2=O


23705it [08:19, 47.46it/s]


In [43]:
with open('min_ld50.pickle', 'wb') as file:
    dump(min_ld50_clear, file)

In [44]:
median_data = {}
for smi, values in tqdm(data.items()):
    if values:
        med = np.median([el['value'] for el in values])
        median_data[smi] = med

100%|██████████| 23726/23726 [00:02<00:00, 8198.17it/s] 


In [45]:
median_ld50 = DataFrame().from_dict({'SMILES': [smi for smi in median_data.keys()], 'LD50': [val for val in median_data.values()]})

In [46]:
median_ld50_clear = DataFrame(columns=['SMILES', 'LD50'])
for _smiles, val in tqdm(zip(median_ld50['SMILES'], median_ld50['LD50'])):
    try:
        mol = ch.smiles(_smiles)
        mol.canonicalize()
        mol.clean_isotopes()
        if mol.check_valence():
            print(f'valerr: {_smiles}')
            continue
        
        median_ld50_clear.loc[len(median_ld50_clear)] = [str(mol), val]
    except Exception as ex:
        print(f'{ex}: {_smiles}')

113it [00:01, 80.34it/s]

valerr: [Al+3].[P-3]
valerr: [P-3].[P-3].[Ca++].[Ca++].[Ca++]
valerr: [Mg++].[Mg++].[Mg++].[P-3].[P-3]


253it [00:05, 31.13it/s]

valerr: [O--].[Cu+].[Cu+]


795it [00:15, 85.59it/s]

...

21175it [06:59, 41.17it/s]

valerr: O.O.O.O.[Pt++].[O-]S([O-])(=O)=O


23188it [07:49, 85.66it/s] 

valerr: [H][C@@](C)(O)C([H])(CCCCCC)[N]1#CN=C2C1=[NH+]C=NC2=O


23705it [08:00, 49.32it/s]


In [47]:
with open('median_ld50.pickle', 'wb') as file:
    dump(median_ld50_clear, file)